# Recensement agricole 2020 — indicateurs par département

**Objectif :** transformer le fichier Agreste `RA2020_001` (format « long », une ligne par indicateur)
en un tableau « large » : **une ligne par département**, une colonne par indicateur.

Ce tableau servira de dénominateur pour l'axe 1 du cadrage (taux de bio = exploitations bio / total).

**Source :** Agreste, tableau RA2020_001, `data/raw/agreste_ra2020/2020/FDS_RA2020_001.zip`
**Sortie :** `data/interim/agreste_ra2020_departements.csv`

In [1]:
from pathlib import Path
import io
import zipfile

import pandas as pd

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SOURCE = RACINE / "data/raw/agreste_ra2020/2020/FDS_RA2020_001.zip"
SORTIE = RACINE / "data/interim/agreste_ra2020_departements.csv"
SOURCE.exists()

True

## 1. Lecture du fichier brut

L'archive contient une seconde archive par millésime (2010 et 2020).
Le CSV 2020 pèse **1,5 Go** décompressé, car il descend jusqu'à la commune (7,4 millions de lignes).

Pour ne pas saturer la mémoire ni le disque, on le lit **par morceaux** de 500 000 lignes
et on ne garde que le niveau département (`GEOGRAPHIE == "DEP"`).

In [2]:
with zipfile.ZipFile(SOURCE) as archive:
    interne = zipfile.ZipFile(io.BytesIO(archive.read("FDS_RA2020_001_2020.zip")))
    with interne.open("FDS_RA2020_001_2020.csv") as fichier:
        morceaux = pd.read_csv(fichier, dtype=str, chunksize=500_000)
        brut = pd.concat(m[m["GEOGRAPHIE"] == "DEP"] for m in morceaux)

brut.shape

(22220, 17)

In [3]:
brut.head()

,NOM,ANNREF,GEOGRAPHIE,CODE_GEO,N122_LIB,N122_MOD,N122,N110d_LIB,N110d_MOD,N110d,RA2020_001_DIM3_LIB,RA2020_001_DIM3_MOD,RA2020_001_DIM3,VALEUR_EXTRAPOLEE,VALEUR_BRUTE,NB_ETAB,IMPUTATION
0,RA2020_001,2020,DEP,01,Classe de taille economique,Ensemble des exploitations,0,OTEX coef17,Toutes exploitations,0,Indicateurs,Nombre d exploitations,001,3017,3017,3017,NON
1,RA2020_001,2020,DEP,01,Classe de taille economique,1.Micro exploitations,0.1,OTEX coef17,Toutes exploitations,0,Indicateurs,Nombre d exploitations,001,889,889,889,NON
2,RA2020_001,2020,DEP,01,Classe de taille economique,2.Petites exploitations,0.2,OTEX coef17,Toutes exploitations,0,Indicateurs,Nombre d exploitations,001,813,813,813,NON
3,RA2020_001,2020,DEP,01,Classe de taille economique,3.Moyennes exploitations,0.3,OTEX coef17,Toutes exploitations,0,Indicateurs,Nombre d exploitations,001,787,787,787,NON
4,RA2020_001,2020,DEP,01,Classe de taille economique,4.Grandes exploitations,0.4,OTEX coef17,Toutes exploitations,0,Indicateurs,Nombre d exploitations,001,528,528,528,NON


## 2. Comprendre la structure

Chaque ligne croise **trois dimensions** et donne une valeur :

| Colonne | Dimension |
|---|---|
| `N122_MOD` | Taille économique de l'exploitation (ensemble, micro, petite, moyenne, grande) |
| `N110d_MOD` | Orientation (OTEX) : grandes cultures, viticulture, maraîchage… ou « Toutes exploitations » |
| `RA2020_001_DIM3_MOD` | Indicateur : nombre d'exploitations, SAU, ETP, production brute standard |
| `VALEUR_EXTRAPOLEE` | La valeur |

In [4]:
for colonne in ["N122_MOD", "N110d_MOD", "RA2020_001_DIM3_MOD"]:
    print(colonne, "→", brut[colonne].nunique(), "valeurs")
    print("   ", sorted(brut[colonne].unique()), "\n")

N122_MOD → 5 valeurs
    ['1.Micro exploitations', '2.Petites exploitations', '3.Moyennes exploitations', '4.Grandes exploitations', 'Ensemble des exploitations'] 

N110d_MOD → 11 valeurs
    ['Exploitations avec ovins et-ou caprins', 'Exploitations bovines - lait', 'Exploitations bovines specialisees - orientation elevage et viande', 'Exploitations bovines specialisees - orientation lait', 'Exploitations de polyculture et-ou polyelevage et exploitations non classees', 'Exploitations specialisees en cultures fruitieres ou autres cultures permanentes', 'Exploitations specialisees en grandes cultures', 'Exploitations specialisees en maraichage ou horticulture', 'Exploitations specialisees en porcins et-ou volailles', 'Exploitations specialisees en viticulture', 'Toutes exploitations'] 

RA2020_001_DIM3_MOD → 4 valeurs
    ['ETP (Equivalent Temps Plein', 'Nombre d exploitations', 'Production brute standard (', 'Superficie agricole utilise'] 



Les valeurs vides correspondent au **secret statistique** : Agreste ne publie pas un chiffre
qui permettrait d'identifier une exploitation (trop peu d'exploitations dans la case).

In [5]:
brut["VALEUR"] = pd.to_numeric(brut["VALEUR_EXTRAPOLEE"])
print("Valeurs sous secret statistique :", brut["VALEUR"].isna().sum(), "sur", len(brut))

Valeurs sous secret statistique : 5409 sur 22220


## 3. Indicateurs globaux par département

On filtre sur l'ensemble des tailles (`N122 == "0"`) et toutes les orientations (`N110d == "0"`),
puis on fait pivoter : les 4 indicateurs deviennent 4 colonnes.

In [6]:
INDICATEURS = {
    "001": "nb_exploitations",
    "002": "sau_ha",
    "003": "etp",
    "004": "pbs_keur",
}

totaux = brut[(brut["N122"] == "0") & (brut["N110d"] == "0")]
print(totaux.groupby("RA2020_001_DIM3")["RA2020_001_DIM3_MOD"].first())

globaux = (
    totaux.assign(indicateur=totaux["RA2020_001_DIM3"].map(INDICATEURS))
    .pivot(index="CODE_GEO", columns="indicateur", values="VALEUR")
    [list(INDICATEURS.values())]
)
globaux.head()

RA2020_001_DIM3
001         Nombre d exploitations
002    Superficie agricole utilise
003    ETP (Equivalent Temps Plein
004    Production brute standard (
Name: RA2020_001_DIM3_MOD, dtype: object


indicateur,nb_exploitations,sau_ha,etp,pbs_keur
CODE_GEO,,,,
01,3017.0,244785.2,5085.087,448136.8
02,4472.0,493926.2,7809.422,1157831.0
03,4356.0,479403.7,5945.853,527521.3
04,2064.0,158946.4,3583.830,255392.5
05,1646.0,91978.8,2770.923,156000.9


## 4. Nombre d'exploitations par orientation (OTEX)

Utile pour l'axe 1 : savoir si un département est plutôt viticole, maraîcher, d'élevage…

In [7]:
OTEX = {
    "Exploitations specialisees en grandes cultures": "grandes_cultures",
    "Exploitations specialisees en maraichage ou horticulture": "maraichage_horticulture",
    "Exploitations specialisees en viticulture": "viticulture",
    "Exploitations specialisees en cultures fruitieres ou autres cultures permanentes": "fruits_cultures_permanentes",
    "Exploitations bovines specialisees - orientation lait": "bovins_lait",
    "Exploitations bovines specialisees - orientation elevage et viande": "bovins_viande",
    "Exploitations bovines - lait": "bovins_mixte",
    "Exploitations avec ovins et-ou caprins": "ovins_caprins",
    "Exploitations specialisees en porcins et-ou volailles": "porcins_volailles",
    "Exploitations de polyculture et-ou polyelevage et exploitations non classees": "polyculture_polyelevage",
}

par_otex = brut[
    (brut["N122"] == "0") & (brut["N110d"] != "0") & (brut["RA2020_001_DIM3"] == "001")
]
otex = (
    par_otex.assign(colonne="nb_expl_" + par_otex["N110d_MOD"].map(OTEX))
    .pivot(index="CODE_GEO", columns="colonne", values="VALEUR")
)
otex.head()

colonne,nb_expl_bovins_lait,nb_expl_bovins_mixte,nb_expl_bovins_viande,nb_expl_fruits_cultures_permanentes,nb_expl_grandes_cultures,nb_expl_maraichage_horticulture,nb_expl_ovins_caprins,nb_expl_polyculture_polyelevage,nb_expl_porcins_volailles,nb_expl_viticulture
CODE_GEO,,,,,,,,,,
01,437.0,55.0,284.0,34.0,1078.0,123.0,253.0,462.0,158.0,133.0
02,301.0,73.0,166.0,33.0,2542.0,78.0,106.0,369.0,68.0,736.0
03,66.0,81.0,1724.0,28.0,936.0,81.0,645.0,469.0,289.0,37.0
04,9.0,10.0,115.0,326.0,557.0,94.0,541.0,349.0,30.0,33.0
05,60.0,38.0,186.0,192.0,246.0,61.0,588.0,221.0,30.0,24.0


## 5. Assemblage et contrôles

In [8]:
ra2020 = globaux.join(otex).reset_index().rename(columns={"CODE_GEO": "code_departement"})
ra2020.insert(1, "annee", 2020)

print("Départements :", len(ra2020))
print("Codes Corse et outre-mer :", [c for c in ra2020["code_departement"] if not c.isdigit() or len(c) == 3])
print("Codes en doublon :", ra2020["code_departement"].duplicated().sum())
print("\nValeurs manquantes (secret statistique) :")
print(ra2020.isna().sum()[lambda s: s > 0])

Départements : 101
Codes Corse et outre-mer : ['2A', '2B', '971', '972', '973', '974', '976']
Codes en doublon : 0

Valeurs manquantes (secret statistique) :
Series([], dtype: int64)


Contrôle de cohérence : la somme des départements doit être proche du total national publié
par Agreste (**389 800 exploitations en France métropolitaine**, environ 416 000 avec l'outre-mer).

In [9]:
print(f"Total exploitations : {ra2020['nb_exploitations'].sum():,.0f}")
print(f"SAU totale : {ra2020['sau_ha'].sum():,.0f} ha")

Total exploitations : 416,436
SAU totale : 26,880,583 ha


In [10]:
ra2020.sort_values("nb_exploitations", ascending=False).head(10)

,code_departement,annee,nb_exploitations,sau_ha,etp,pbs_keur,nb_expl_bovins_lait,nb_expl_bovins_mixte,nb_expl_bovins_viande,nb_expl_fruits_cultures_permanentes,nb_expl_grandes_cultures,nb_expl_maraichage_horticulture,nb_expl_ovins_caprins,nb_expl_polyculture_polyelevage,nb_expl_porcins_volailles,nb_expl_viticulture
51,51,2020,13231.0,557179.30,19877.700,2685717.0,40.0,7.0,54.0,20.0,3063.0,94.0,29.0,554.0,65.0,9305.0
64,64,2020,9784.0,322742.80,11569.620,752567.3,314.0,119.0,1318.0,114.0,3176.0,216.0,2524.0,1150.0,545.0,308.0
50,50,2020,8038.0,420871.70,11218.540,1213290.0,2334.0,385.0,1635.0,96.0,1391.0,235.0,1230.0,404.0,327.0,1.0
34,34,2020,7891.0,175789.70,10799.390,670948.0,1.0,6.0,86.0,402.0,222.0,276.0,299.0,285.0,39.0,6275.0
11,12,2020,7637.0,508957.00,11216.640,695968.2,665.0,212.0,2723.0,83.0,1001.0,94.0,2142.0,414.0,233.0,70.0
20,22,2020,7316.0,433338.30,13791.930,2187046.0,1845.0,131.0,497.0,91.0,1439.0,333.0,270.0,705.0,2005.0,0.0
96,971,2020,7254.0,31835.85,6440.196,162812.4,0.0,0.0,2141.0,571.0,2163.0,473.0,159.0,1458.0,289.0,0.0
33,33,2020,7023.0,233393.70,24808.930,2245473.0,27.0,13.0,211.0,94.0,488.0,261.0,203.0,275.0,73.0,5378.0
35,35,2020,7001.0,440456.00,12638.800,1533633.0,2503.0,198.0,666.0,120.0,1268.0,285.0,361.0,740.0,858.0,2.0
32,32,2020,6680.0,448499.20,8419.630,714663.4,51.0,42.0,352.0,79.0,3692.0,113.0,276.0,1044.0,456.0,575.0


## 6. Sauvegarde

Le résultat va dans `data/interim/` : c'est un fichier de travail.
Dans le pipeline final, cette transformation sera faite en couche **silver**.

In [11]:
SORTIE.parent.mkdir(parents=True, exist_ok=True)
ra2020.to_csv(SORTIE, index=False, encoding="utf-8")
print(SORTIE, "→", ra2020.shape)

F:\Projets_Data\bio-operators-etl-pipeline\data\interim\agreste_ra2020_departements.csv → (101, 16)
